# T23b — Fine-tune bge-base-en-v1.5 for CallVerse Intake (Kaggle, GPU)

Written by the Brain, 2026-10-31, approved by Yassine. **Settings: Accelerator = GPU T4 (or P100), Internet = ON.** Then *Run all*.

What it does: builds training messages from **ABCD `train` only** (first customer message with ≥ 4 words, the same subflow → reason mapping as T23) + CallVerse's own reason examples (ShopNova 60, Atlas 80), fine-tunes `BAAI/bge-base-en-v1.5` with a label-based triplet loss so messages with the same reason move closer, and saves the model as a zip to download.

**Never used here:** ABCD `dev` / `test`, the Atlas val / test sets, the rule-search sets. Evaluation happens back on the PC with the T23 benchmark code.

In [ ]:
!pip -q install "sentence-transformers>=3.0,<7" datasets
import json, gzip, random, re, urllib.request, os, shutil, collections
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
SEED = 2026
random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 1. Data (public sources only)
REPO = "https://raw.githubusercontent.com/yassinemaazaoui114-git/callverse/feature/t23-m3-benchmark/"
def get_json(url):
    with urllib.request.urlopen(url) as r: return json.loads(r.read().decode("utf-8"))
urllib.request.urlretrieve("https://raw.githubusercontent.com/asappresearch/abcd/master/data/abcd_v1.1.json.gz", "abcd.json.gz")
abcd = json.load(gzip.open("abcd.json.gz", "rt", encoding="utf-8"))
mapping = get_json(REPO + "benchmarks/m3/data/abcd_mapping.json")
shop_ex = get_json(REPO + "callverse/intake/data/reason_examples.json")
atlas_ex = get_json(REPO + "callverse/intake/data/reason_examples_atlas.json")
print({k: len(v) for k, v in abcd.items()})

In [ ]:
# 2. Training set: ABCD TRAIN only, same rules as T23 (first customer turn with >= 4 words)
sub2reason = {}
for reason, subs in mapping.items():
    if isinstance(subs, list):
        for s in subs: sub2reason[s] = reason
def first_turn(chat):
    for who, text in chat["original"]:
        if who == "customer" and len(text.split()) >= 4: return text.strip()
    return None
rows = collections.defaultdict(list)
for chat in abcd["train"]:
    t = first_turn(chat)
    if t: rows["shopnova:" + sub2reason.get(chat["scenario"]["subflow"], "other")].append(t)
OTHER_CAP = 1500   # 'other' is by far the largest class: keep 1,500 (seed 2026)
random.shuffle(rows["shopnova:other"]); rows["shopnova:other"] = rows["shopnova:other"][:OTHER_CAP]
for reason, msgs in shop_ex.items():
    if not reason.startswith("_"): rows["shopnova:" + reason] += msgs
for reason, msgs in atlas_ex.items():
    if not reason.startswith("_"): rows["atlas:" + reason] += msgs
labels = sorted(rows); label_id = {l: i for i, l in enumerate(labels)}
data = [(t, label_id[l]) for l in labels for t in rows[l]]
random.shuffle(data)
print(len(data), {l: len(rows[l]) for l in labels})

In [ ]:
# 3. Fine-tune (label-based triplet loss; batch 32, 2 epochs, lr 2e-5, warm-up 10 %, fp16)
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
train = Dataset.from_dict({"sentence": [t for t, _ in data], "label": [l for _, l in data]})
loss = losses.BatchAllTripletLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir="out", num_train_epochs=2, per_device_train_batch_size=32, learning_rate=2e-5,
    warmup_ratio=0.1, fp16=torch.cuda.is_available(), seed=SEED, logging_steps=50, save_strategy="no",
    batch_sampler=BatchSamplers.GROUP_BY_LABEL, report_to="none")
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=loss)
trainer.train()

In [ ]:
# 4. Quick sanity check on TRAINING data only (not a score): same-reason pairs should be closer than other pairs
import numpy as np
sample = random.sample(data, 400)
emb = model.encode([t for t, _ in sample], normalize_embeddings=True)
lab = np.array([l for _, l in sample]); sim = emb @ emb.T
same = sim[(lab[:, None] == lab[None, :]) & ~np.eye(len(lab), dtype=bool)].mean()
diff = sim[lab[:, None] != lab[None, :]].mean()
print(f"mean cosine same reason {same:.3f} vs different reason {diff:.3f}")

In [ ]:
# 5. Save + zip for download (Output tab -> bge-base-callverse-ft.zip)
model.save("bge-base-callverse-ft")
json.dump({"base": "BAAI/bge-base-en-v1.5", "train_rows": len(data), "labels": labels, "seed": SEED,
           "epochs": 2, "batch": 32, "lr": 2e-5, "loss": "BatchAllTripletLoss", "data": "ABCD train (MIT) + CallVerse examples"},
          open("bge-base-callverse-ft/callverse_training.json", "w"), indent=1)
shutil.make_archive("bge-base-callverse-ft", "zip", "bge-base-callverse-ft")
print(os.path.getsize("bge-base-callverse-ft.zip") // 1_000_000, "MB")